<a href="https://colab.research.google.com/github/talhanoor23/generative-ai/blob/main/NLP/text_pre_Word2vec_Ran_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

we can do preprocessing from scratch or we can use simple_preprocess function. we can train word2vec from scratch or we can use pretrained word2vec from google. we can use word2vec or embeddings models.


## **Import_Datasets**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import tensorflow as tf
import numpy as np
import matplotlib
import plotly.express as px
import os
%matplotlib inline

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
 62% 16.0M/25.7M [00:00<00:00, 164MB/s]
100% 25.7M/25.7M [00:00<00:00, 201MB/s]


In [ ]:
!unzip imdb-dataset-of-50k-movie-reviews.zip

Archive:  imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        


In [ ]:
df = pd.read_csv('IMDB Dataset.csv')

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **Preprocessing_From_Scratch**

In [ ]:
!pip install autocorrect
!pip install contractions

In [ ]:
import re
import string
import spacy
import nltk
import gensim
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize
from autocorrect import Speller
from contractions import fix

In [ ]:

# Load Spacy Model
nlp = spacy.load("en_core_web_sm")
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
#no need to use them bcz we use "nlp=spacy.../nlp.pipe" which automatically word tokenize, check stop words, and lemmatize the words using
    # docs = list(nlp.pipe(cleaned_sentences, batch_size=50))  # Batch processing in spaCy
    # return [[token.lemma_ for token in doc if not token.is_stop] for doc in docs]
# we donot need to use nltk speratly.

# stemmer = PorterStemmer()
# lemmatizer = WordNetLemmatizer()
# stop_words = set(stopwords.words('english'))
# spell = Speller(lang='en')

In [ ]:
# Load optimized SpaCy model (disable unnecessary components for speed)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def clean_text(text):
    """Preprocess and clean a single sentence."""
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = fix(text)  # Expand contractions
    text = text.lower().strip()  # Convert to lowercase
    return text

def process_sentences(text):
    """Tokenizes text into sentences, cleans each sentence, and processes in batch."""
    sentences = sent_tokenize(text)  # Split into sentences
    cleaned_sentences = [clean_text(sent) for sent in sentences]  # Clean each sentence
    docs = list(nlp.pipe(cleaned_sentences, batch_size=50))  # Batch processing in spaCy
    return [[token.lemma_ for token in doc if not token.is_stop] for doc in docs]  # Tokenize & lemmatize

In [ ]:
# Apply processing to each row using `.apply()`
df["processed_sentences"] = df["review"].apply(process_sentences)

# Flatten all sentences into a single list for Word2Vec
#each row has list containing lists of tokenize sentences. sentences fetch the whole list of firstrow, sentence fetch the lists of sentence within the row list.
all_sentences = [sentence for sentences in df["processed_sentences"] for sentence in sentences]

In [ ]:
#but we use .apply and sent_tokenize inside the function above. so no need to use this.
# story = []
# for doc in df['review']:
#     raw_sent = sent_tokenize(doc)
#     for sent in raw_sent:
#         story.append(clean_text(sent))

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
df['sentiment_enc'] = encoder.fit_transform(df['sentiment'])

In [ ]:
df["processed_sentences"][0]

# **Train_Test_Split**

In [ ]:
# df.to_csv('word2vec_df.csv', index=False)

In [ ]:
df

,review,sentiment,processed_sentences,sentiment_enc
0,One of the other reviewers has mentioned that ...,positive,"[[reviewer, mention, watch, 1, oz, episode, ho...",1
1,A wonderful little production. <br /><br />The...,positive,"[[wonderful, little, production, .], [film, te...",1
2,I thought this was a wonderful way to spend ti...,positive,"[[think, wonderful, way, spend, time, hot, sum...",1
3,Basically there's a family where a little boy ...,negative,"[[basically, family, little, boy, (, jake, ), ...",0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"[[petter, mattei, "", love, time, money, "", vis...",1
...,...,...,...,...
49995,I thought this movie did a down right good job...,positive,"[[think, movie, right, good, job, .], [creativ...",1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,"[[bad, plot, ,, bad, dialogue, ,, bad, act, ,,...",0
49997,I am a Catholic taught in parochial elementary...,negative,"[[catholic, teach, parochial, elementary, scho...",0
49998,I'm going to have to disagree with the previou...,negative,"[[go, disagree, previous, comment, maltin, .],...",0


In [ ]:
X = df['processed_sentences']
y = df['sentiment_enc']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **word2vec_Training**

In [ ]:
from gensim.models import Word2Vec

#easy wording and automatic. vocab_building and train occur automatically in one line of code.
Word2Vec_model = Word2Vec(sentences=all_sentences, vector_size=100, window=5, min_count=1, workers=4)

In [ ]:
#vocab_building and train occur step by step and manually. more control over epocs and corpus. but work same as above code.

# model = gensim.models.Word2Vec(
#     window=10,
#     min_count=2
# )
# model.build_vocab(all_sentences)
# model.train(all_sentences, total_examples=model.corpus_count, epochs=model.epochs)

In [ ]:
Word2Vec_model.save("word2vec.model")

# loaded_model = Word2Vec.load("word2vec.model")